# SENSE 프로젝트 — TimesFM 2.5 시계열 예측 노트북

SENSE 프로젝트 — TimesFM 2.5 시계열 예측 노트북
SENSE: Semiconductor Economic News & Signal Engine

목적: TimesFM 2.5를 활용한 반도체 주가 시계열 예측
      Step 1 — Zero-shot Baseline (주가만)
      Step 2 — XReg 공변량 추론 (매크로/퀀트/감성)
      Step 3 — 매크로 충격 정량화 + 시나리오 분석 + Backtest

실행 환경: Databricks GPU 클러스터 권장 (Standard_NC6s_v3 이상)
          CPU에서도 동작하나 배치 추론 시 느림

전략 문서: ref/TimesFM.md
데이터 사전: docs/data_dict/

# 0. 환경 설정 및 패키지 설치

In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%sh
uv pip install "timesfm[torch] @ git+https://github.com/google-research/timesfm.git" openai "jax>=0.5,<0.6" "jaxlib>=0.5,<0.6" --upgrade
echo "--- packages installed, restart python kernel ---"

Using Python 3.12.3 environment at: /local_disk0/.ephemeral_nfs/envs/pythonEnv-ac54de74-59d0-4ad2-9642-4e45defa2fae
Resolved 64 packages in 738ms
   Updating https://github.com/google-research/timesfm.git (HEAD)
 Downloaded nvidia-cufile
 Downloaded pydantic-core
 Downloaded nvidia-cuda-runtime
 Downloaded pygments
 Downloaded setuptools
 Downloaded hf-xet
 Downloaded jax
    Updated https://github.com/google-research/timesfm.git (f085b9079918092aa5e3917a4e135f87f91a7f03)
   Building timesfm @ git+https://github.com/google-research/timesfm.git@f085b9079918092aa5e3917a4e135f87f91a7f03
 Downloaded ml-dtypes
 Downloaded networkx
 Downloaded cuda-bindings
 Downloaded openai
 Downloaded sympy
      Built timesfm @ git+https://github.com/google-research/timesfm.git@f085b9079918092aa5e3917a4e135f87f91a7f03
 Downloaded nvidia-cuda-cupti
 Downloaded numpy
 Downloaded scipy
 Downloaded nvidia-nvjitlink
 Downloaded nvidia-curand
 Downloaded nvidia-nvshmem-cu13
 Downloaded nvidia-cuda-nvrtc
 Downl

--- packages installed, restart python kernel ---


In [0]:
import os
import sys
import warnings

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore", category=FutureWarning)

REPO_PATH = "/Workspace/Repos/3dt005@msacademy.msai.kr/3dt-2nd-project"
sys.path.insert(0, f"{REPO_PATH}/src")

os.environ["KEY_VAULT_URL"] = "https://kv-3dt-team1.vault.azure.net/"

# 1. ADLS Gen2 연결 및 데이터 로드

In [0]:
from utils.vault_manager import get_vault_manager  # noqa: E402

vault = get_vault_manager()
vault.get_storage_client()  # Spark OAuth conf 자동 설정
account = vault.get_secret("adls-account-name")  # "3dtteam1adls"

[OK] Key Vault 클라이언트 연결 완료: https://kv-3dt-team1.vault.azure.net/
[OK] Spark conf ADLS Gen2 OAuth 설정 완료: 3dtteam1adls


## 1-0. Azure OpenAI 클라이언트 초기화

In [0]:
from openai import AzureOpenAI  # noqa: E402

OPENAI_DEPLOYMENT = "gpt-4.1-mini"
OPENAI_API_VERSION = "2025-03-01-preview"

_openai_endpoint = vault.get_secret("azure-openai-endpoint")
_openai_key = vault.get_secret("azure-openai-key")

openai_client = AzureOpenAI(
    azure_endpoint=_openai_endpoint,
    api_key=_openai_key,
    api_version=OPENAI_API_VERSION,
)

_SYSTEM_MSG = (
    "당신은 반도체 주식 시장 전문 퀀트 애널리스트입니다. "
    "TimesFM 시계열 예측 결과를 바탕으로 한국어로 간결하고 "
    "전문적인 투자 인사이트를 제공합니다. "
    "수치 근거를 반드시 포함하고, 리스크도 균형있게 언급하세요."
)


def ask_gpt(prompt: str, system_msg: str | None = None, max_tokens: int = 1200) -> str:
    """Azure OpenAI GPT에 프롬프트를 보내고 응답 문자열을 반환합니다."""
    messages = []
    if system_msg:
        messages.append({"role": "system", "content": system_msg})
    messages.append({"role": "user", "content": prompt})
    try:
        resp = openai_client.chat.completions.create(
            model=OPENAI_DEPLOYMENT,
            messages=messages,
            max_tokens=max_tokens,
            temperature=0.3,
        )
        return resp.choices[0].message.content or ""
    except Exception as e:  # noqa: BLE001
        return f"[OpenAI 오류] {e}"


print(f"Azure OpenAI 연결 완료 | 엔드포인트: {_openai_endpoint} | 배포: {OPENAI_DEPLOYMENT}")

Azure OpenAI 연결 완료 | 엔드포인트: https://aoai-3dt-team1.openai.azure.com/ | 배포: gpt-4.1-mini


## 1-1. 타겟 시계열 — 주가 OHLCV

In [0]:
TICKERS = ["005930.KS", "000660.KS"]
TICKER_NAMES = {"005930.KS": "삼성전자", "000660.KS": "SK하이닉스"}
HORIZON = 20

TICKER_COL_MAP = {
    "005930.KS": "yfinance_samsung_close",
    "000660.KS": "yfinance_skhynix_close",
}

from pyspark.sql.utils import AnalysisException  # noqa: E402

def safe_read_parquet(path, date_col=None, index_col=None):
    try:
        df = spark.read.parquet(path).toPandas()
        if date_col and date_col in df.columns:
            df[date_col] = pd.to_datetime(df[date_col])
        if index_col and index_col in df.columns:
            df = df.set_index(index_col).sort_index()
        label = path.split('/')[-1]
        print(f"  [OK] {label}: {len(df)} rows, cols={list(df.columns)[:8]}...")
        return df
    except Exception as e:
        print(f"  [SKIP] {path.split('/')[-1]} ({type(e).__name__})")
        return pd.DataFrame()

# ---------------------------------------------------------------------------
# 1-1. 주가 + 매크로 데이터 로드 (feature/curated 컨테이너 탐색)
# ---------------------------------------------------------------------------
_gold_paths = [
    f"abfss://feature@{account}.dfs.core.windows.net/gold_macro_1y.parquet",
    f"abfss://curated@{account}.dfs.core.windows.net/gold_macro_1y.parquet",
    f"abfss://curated@{account}.dfs.core.windows.net/pre_macro_1y_adf.parquet",
]

_df_gold = pd.DataFrame()
for _path in _gold_paths:
    _df_gold = safe_read_parquet(_path)
    if not _df_gold.empty:
        break

if not _df_gold.empty:
    # 날짜 컬럼 자동감지
    _date_candidates = ["기준일자", "trade_date", "date", "Date"]
    _date_col = next((c for c in _date_candidates if c in _df_gold.columns), None)
    if _date_col is None:
        # datetime 타입 컬럼 탐색
        for c in _df_gold.columns:
            if "date" in c.lower() or "_dt" in c.lower() or "일자" in c:
                _date_col = c
                break
    if _date_col is None:
        print(f"[WARN] 날짜 컬럼 미발견. 컬럼들: {list(_df_gold.columns)}")
        _df_gold = pd.DataFrame()

if not _df_gold.empty:
    _df_gold[_date_col] = pd.to_datetime(_df_gold[_date_col])
    _df_gold = _df_gold.drop_duplicates(subset=[_date_col], keep="first").sort_values(_date_col)
    if _date_col != "trade_date":
        _df_gold = _df_gold.rename(columns={_date_col: "trade_date"})

    # (A) 주가 타겟
    _has_holidays = "주말여부" in _df_gold.columns and "한국_휴장일_여부" in _df_gold.columns
    _df_trading = _df_gold[(~_df_gold["주말여부"]) & (~_df_gold["한국_휴장일_여부"])].copy() if _has_holidays else _df_gold.copy()

    equity_records = []
    for ticker, col_name in TICKER_COL_MAP.items():
        if col_name in _df_trading.columns:
            _sub = _df_trading[["trade_date", col_name]].dropna(subset=[col_name]).copy()
            _sub = _sub.rename(columns={col_name: "close"})
            _sub["ticker"] = ticker
            equity_records.append(_sub[["ticker", "trade_date", "close"]])
    df_equity = pd.concat(equity_records, ignore_index=True).sort_values(["ticker", "trade_date"]).reset_index(drop=True) if equity_records else pd.DataFrame(columns=["ticker", "trade_date", "close"])

    # (B) 매크로 피처
    _macro_cols = ["usd_krw_rate", "yfinance_nvda_close", "yfinance_amd_close", "yfinance_mu_close",
                   "yfinance_tsm_close", "yfinance_asml_close", "yfinance_sox_close",
                   "fred_dff", "fred_dgs10", "fred_dgs2", "fred_t10y2y", "fred_dfii10", "fred_bamlh0a0hym2"]
    _macro_avail = [c for c in _macro_cols if c in _df_gold.columns]
    df_macro_gold = _df_gold.set_index("trade_date")[_macro_avail].sort_index().ffill().bfill() if _macro_avail else pd.DataFrame()
    if not df_macro_gold.empty:
        df_macro_gold.index.name = "trade_date"

    print(f"\n주가: {len(df_equity)} rows | 매크로: {df_macro_gold.shape[1] if not df_macro_gold.empty else 0} cols")
    for t in TICKERS:
        print(f"  {TICKER_NAMES[t]}: {len(df_equity[df_equity['ticker']==t])} 거래일")
else:
    df_equity = pd.DataFrame(columns=["ticker", "trade_date", "close"])
    df_macro_gold = pd.DataFrame()
    print("[WARN] 주가/매크로 데이터 로드 실패")

  [SKIP] gold_macro_1y.parquet (AnalysisException)
  [SKIP] gold_macro_1y.parquet (AnalysisException)
  [OK] pre_macro_1y_adf.parquet: 475 rows, cols=['date', 'usd_krw_rate', 'yfinance_amd_close', 'yfinance_asml_close', 'yfinance_intc_close', 'yfinance_mu_close', 'yfinance_nvda_close', 'yfinance_samsung_close']...

주가: 486 rows | 매크로: 13 cols
  삼성전자: 243 거래일
  SK하이닉스: 243 거래일


## 1-2. 한국 금융 파생상품 피처 (silver_kfinance)

In [0]:
# ---------------------------------------------------------------------------
# 1-2. silver_kfinance.parquet → 한국 금융 파생상품 피처
#  KOSPI200 옵션/워런트 월별 데이터 → 일별 Forward-Fill
#  피처: ATM가, 활성계약수, 평균가, 최대유효행사가, 총가치
# ---------------------------------------------------------------------------
kfin_path = f"abfss://curated@{account}.dfs.core.windows.net/silver_kfinance.parquet"
_df_kfin_raw = safe_read_parquet(kfin_path, date_col="date")

if not _df_kfin_raw.empty:
    # close_price를 숫자로 변환
    _df_kfin_raw["close_price"] = pd.to_numeric(_df_kfin_raw["close_price"], errors="coerce").fillna(0)

    # ticker에서 행사가 추출 (kfinance_201WC170 → 170)
    _df_kfin_raw["strike"] = (
        _df_kfin_raw["ticker"]
        .str.extract(r"(\d+)$")[0]
        .astype(float)
    )

    # 날짜별 집계 → 시장 레벨 피처
    df_kfin_agg = (
        _df_kfin_raw.groupby("date")
        .agg(
            # ATM(최고가) 옵션 가격 → 변동성 프록시
            kfin_atm_price=("close_price", "max"),
            # 활성 계약 수 (close_price > 0) → 시장 폭
            kfin_active_count=("close_price", lambda x: (x > 0).sum()),
            # 활성 옵션 평균 가격
            kfin_mean_price=("close_price", lambda x: x[x > 0].mean() if (x > 0).any() else 0),
            # 총 가치 합계 → 시장 활동 강도
            kfin_total_value=("close_price", "sum"),
            # 전체 종목 수
            kfin_total_count=("close_price", "count"),
        )
        .reset_index()
    )

    # 최대 유효 행사가 (close > 0인 최고 행사가) → 시장 상한 기대
    _active = _df_kfin_raw[_df_kfin_raw["close_price"] > 0]
    _max_strike = _active.groupby("date")["strike"].max().reset_index()
    _max_strike.columns = ["date", "kfin_max_strike"]
    df_kfin_agg = df_kfin_agg.merge(_max_strike, on="date", how="left")

    # 활성 비율
    df_kfin_agg["kfin_active_ratio"] = df_kfin_agg["kfin_active_count"] / df_kfin_agg["kfin_total_count"]

    # 월별 → 일별 Forward-Fill
    df_kfin_agg = df_kfin_agg.set_index("date").sort_index()
    all_bdays = pd.bdate_range(df_kfin_agg.index.min(), df_kfin_agg.index.max(), freq="B")
    df_kfinance = df_kfin_agg.reindex(all_bdays).ffill().bfill()
    df_kfinance.index.name = "trade_date"

    # 불필요 컬럼 제거
    df_kfinance = df_kfinance.drop(columns=["kfin_total_count"], errors="ignore")

    print(f"kfinance 피처: {df_kfinance.shape}")
    print(f"  기간: {df_kfinance.index.min()} ~ {df_kfinance.index.max()}")
    print(f"  컬럼: {list(df_kfinance.columns)}")
    display(df_kfinance.head())
else:
    df_kfinance = pd.DataFrame()
    print("[WARN] kfinance 데이터 없음")

  [OK] silver_kfinance.parquet: 140000 rows
kfinance 피처: (571, 6)
  기간: 2024-01-30 00:00:00 ~ 2026-04-07 00:00:00
  컬럼: ['kfin_atm_price', 'kfin_active_count', 'kfin_mean_price', 'kfin_total_value', 'kfin_max_strike', 'kfin_active_ratio']


kfin_atm_price,kfin_active_count,kfin_mean_price,kfin_total_value,kfin_max_strike,kfin_active_ratio
54000.0,533.0,1701.5132457786115,906906.56,27367.0,0.1066
54000.0,533.0,1701.5132457786115,906906.56,27367.0,0.1066
54000.0,533.0,1701.5132457786115,906906.56,27367.0,0.1066
54000.0,533.0,1701.5132457786115,906906.56,27367.0,0.1066
54000.0,533.0,1701.5132457786115,906906.56,27367.0,0.1066


## 1-3. 반도체 수출입 피처 (silver_semiconductor)

In [0]:
# ---------------------------------------------------------------------------
# 1-3. silver_semiconductor.parquet → 반도체 수출입 피처
#  월별 HS코드별 수출/수입 → 일별 Forward-Fill
#  피처: 총수출, 총수입, 무역수지, DRAM수출, Flash수출, MoM변화, DRAM비중
# ---------------------------------------------------------------------------
semi_path = f"abfss://curated@{account}.dfs.core.windows.net/silver_semiconductor.parquet"
_df_semi_raw = safe_read_parquet(semi_path, date_col="date")

if not _df_semi_raw.empty:
    # --- 날짜별 전체 집계 ---
    _semi_total = (
        _df_semi_raw.groupby("date")
        .agg(semi_total_exp=("expDlr", "sum"), semi_total_imp=("impDlr", "sum"))
        .reset_index()
    )
    _semi_total["semi_net_trade"] = _semi_total["semi_total_exp"] - _semi_total["semi_total_imp"]

    # --- DRAM (HS 8542321010) ---
    _dram = _df_semi_raw[_df_semi_raw["hsCode"] == 8542321010].groupby("date").agg(
        semi_dram_exp=("expDlr", "sum"), semi_dram_imp=("impDlr", "sum")
    ).reset_index()

    # --- Flash 메모리 (HS 8542321030) ---
    _flash = _df_semi_raw[_df_semi_raw["hsCode"] == 8542321030].groupby("date").agg(
        semi_flash_exp=("expDlr", "sum"), semi_flash_imp=("impDlr", "sum")
    ).reset_index()

    # --- 복합구조칩 IC (HS 8542323000, 최대 수출 품목) ---
    _mcp = _df_semi_raw[_df_semi_raw["hsCode"] == 8542323000].groupby("date").agg(
        semi_mcp_exp=("expDlr", "sum")
    ).reset_index()

    # 병합
    df_semi_agg = _semi_total
    for _sub in [_dram, _flash, _mcp]:
        df_semi_agg = df_semi_agg.merge(_sub, on="date", how="left")

    df_semi_agg = df_semi_agg.sort_values("date").reset_index(drop=True)

    # --- 파생 피처 ---
    # DRAM 비중 (전체 반도체 수출 대비)
    df_semi_agg["semi_dram_ratio"] = df_semi_agg["semi_dram_exp"] / df_semi_agg["semi_total_exp"].replace(0, np.nan)
    # MoM 변화율 ()
    df_semi_agg["semi_exp_mom"] = df_semi_agg["semi_total_exp"].pct_change()
    df_semi_agg["semi_dram_mom"] = df_semi_agg["semi_dram_exp"].pct_change()
    # 무역수지 비율 (수출/수입)
    df_semi_agg["semi_trade_ratio"] = df_semi_agg["semi_total_exp"] / df_semi_agg["semi_total_imp"].replace(0, np.nan)

    # 금액 단위 조정 (USD → 억 USD)
    dollar_cols = [c for c in df_semi_agg.columns if c.startswith("semi_") and ("exp" in c or "imp" in c or "net" in c) and "mom" not in c and "ratio" not in c]
    for col in dollar_cols:
        df_semi_agg[col] = df_semi_agg[col] / 1e8  # 억달러 단위

    # 월별 → 일별 Forward-Fill
    df_semi_agg = df_semi_agg.set_index("date").sort_index()
    all_bdays = pd.bdate_range(df_semi_agg.index.min(), df_semi_agg.index.max(), freq="B")
    df_semiconductor = df_semi_agg.reindex(all_bdays).ffill().bfill()
    df_semiconductor.index.name = "trade_date"

    print(f"semiconductor 피처: {df_semiconductor.shape}")
    print(f"  기간: {df_semiconductor.index.min()} ~ {df_semiconductor.index.max()}")
    print(f"  컬럼: {list(df_semiconductor.columns)}")
    display(df_semiconductor.head())
else:
    df_semiconductor = pd.DataFrame()
    print("[WARN] semiconductor 데이터 없음")

  [OK] silver_semiconductor.parquet: 520 rows
semiconductor 피처: (545, 12)
  기간: 2024-01-01 00:00:00 ~ 2026-01-30 00:00:00
  컬럼: ['semi_total_exp', 'semi_total_imp', 'semi_net_trade', 'semi_dram_exp', 'semi_dram_imp', 'semi_flash_exp', 'semi_flash_imp', 'semi_mcp_exp', 'semi_dram_ratio', 'semi_exp_mom', 'semi_dram_mom', 'semi_trade_ratio']


semi_total_exp,semi_total_imp,semi_net_trade,semi_dram_exp,semi_dram_imp,semi_flash_exp,semi_flash_imp,semi_mcp_exp,semi_dram_ratio,semi_exp_mom,semi_dram_mom,semi_trade_ratio
190.41615494,102.23963614,88.1765188,33.19242989,11.39582369,12.97983363,3.6728651,54.43177078,0.17431519873121537,-0.5473366864951148,-0.5506620316913472,1.8624494582439346
190.41615494,102.23963614,88.1765188,33.19242989,11.39582369,12.97983363,3.6728651,54.43177078,0.17431519873121537,-0.5473366864951148,-0.5506620316913472,1.8624494582439346
190.41615494,102.23963614,88.1765188,33.19242989,11.39582369,12.97983363,3.6728651,54.43177078,0.17431519873121537,-0.5473366864951148,-0.5506620316913472,1.8624494582439346
190.41615494,102.23963614,88.1765188,33.19242989,11.39582369,12.97983363,3.6728651,54.43177078,0.17431519873121537,-0.5473366864951148,-0.5506620316913472,1.8624494582439346
190.41615494,102.23963614,88.1765188,33.19242989,11.39582369,12.97983363,3.6728651,54.43177078,0.17431519873121537,-0.5473366864951148,-0.5506620316913472,1.8624494582439346


## 1-4. 피처 요약

In [0]:
# ---------------------------------------------------------------------------
# 1-4. curated 피처 요약
# ---------------------------------------------------------------------------
print("=" * 60)
print("SENSE Feature 요약 (curated 기반)")
print("=" * 60)

if not df_kfinance.empty:
    print(f"\n[한국 금융 파생상품]  {df_kfinance.shape[1]} 피처, {len(df_kfinance)} 일")
    print(f"  기간: {df_kfinance.index.min().date()} ~ {df_kfinance.index.max().date()}")
    print(f"  피처: {list(df_kfinance.columns)}")
else:
    print("\n[한국 금융] 데이터 없음")

if not df_semiconductor.empty:
    print(f"\n[반도체 수출입]  {df_semiconductor.shape[1]} 피처, {len(df_semiconductor)} 일")
    print(f"  기간: {df_semiconductor.index.min().date()} ~ {df_semiconductor.index.max().date()}")
    print(f"  피처: {list(df_semiconductor.columns)}")
else:
    print("\n[반도체] 데이터 없음")

if not df_equity.empty:
    print(f"\n[주가 타겟]  {len(df_equity)} rows")
    print(f"  종목: {[TICKER_NAMES[t] for t in TICKERS]}")
else:
    print("\n[주가] 데이터 없음")

print("\n" + "=" * 60)

SENSE Feature 요약 (curated 기반)

[한국 금융 파생상품]  6 피처, 571 일
  기간: 2024-01-30 ~ 2026-04-07
  피처: ['kfin_atm_price', 'kfin_active_count', 'kfin_mean_price', 'kfin_total_value', 'kfin_max_strike', 'kfin_active_ratio']

[반도체 수출입]  12 피처, 545 일
  기간: 2024-01-01 ~ 2026-01-30
  피처: ['semi_total_exp', 'semi_total_imp', 'semi_net_trade', 'semi_dram_exp', 'semi_dram_imp', 'semi_flash_exp', 'semi_flash_imp', 'semi_mcp_exp', 'semi_dram_ratio', 'semi_exp_mom', 'semi_dram_mom', 'semi_trade_ratio']

[주가] 데이터 없음



# 2. 통합 피처 마트 구성

In [0]:

def build_feature_mart(df_equity, ticker, df_kfinance, df_semiconductor, df_macro_gold):
    """
    종목별 통합 피처 마트를 date 기준으로 LEFT JOIN하여 구성합니다.
    curated: kfinance(한국 금융) + semiconductor(수출입) + macro_gold(FRED/FX/피어주)
    """
    df = df_equity[df_equity["ticker"] == ticker][["trade_date", "close"]].copy()
    df = df.set_index("trade_date").sort_index()

    # 수익률 파생
    df["return_1d"] = df["close"].pct_change()

    # --- 한국 금융 파생상품 피처 ---
    if not df_kfinance.empty:
        df = df.join(df_kfinance, how="left")

    # --- 반도체 수출입 피처 ---
    if not df_semiconductor.empty:
        df = df.join(df_semiconductor, how="left")

    # --- 매크로/글로벌 피어 피처 ---
    if not df_macro_gold.empty:
        df = df.join(df_macro_gold, how="left")

    # Forward fill 후 첫 행 NaN 제거
    df = df.ffill().bfill()

    return df


# 종목별 피처 마트 생성
feature_marts = {}
for ticker in TICKERS:
    feature_marts[ticker] = build_feature_mart(
        df_equity, ticker, df_kfinance, df_semiconductor, df_macro_gold
    )
    print(f"{TICKER_NAMES[ticker]}: {feature_marts[ticker].shape}")

삼성전자: (243, 33)
SK하이닉스: (243, 33)


# 3. 교차 검증 파생 변수 생성

In [0]:

def create_derived_features(df):
    """
    SENSE 3축 교차 검증 파생 변수 + 시차/변동성 피처를 생성합니다.
    curated 데이터 기반: kfinance(금융 파생) × semiconductor(수출입)
    """
    out = df.copy()

    # === 교호 작용 파생 변수 ===

    # 금융 × 수출: 옵션 시장 활성도 × 수출 모멘텀
    if "kfin_active_ratio" in out.columns and "semi_exp_mom" in out.columns:
        out["finance_export_synergy"] = out["kfin_active_ratio"] * out["semi_exp_mom"].fillna(0)

    # 옵션 ATM가 × DRAM 수출: 시장 기대 × DRAM 실적
    if "kfin_atm_price" in out.columns and "semi_dram_exp" in out.columns:
        out["atm_dram_cross"] = (
            out["kfin_atm_price"] / out["kfin_atm_price"].rolling(5, min_periods=1).mean()
        ) * (
            out["semi_dram_exp"] / out["semi_dram_exp"].rolling(3, min_periods=1).mean()
        )

    # 무역수지 × 옵션 총가치: 실물 × 금융 복합 지표
    if "semi_net_trade" in out.columns and "kfin_total_value" in out.columns:
        out["trade_finance_compound"] = (
            out["semi_net_trade"] / out["semi_net_trade"].abs().rolling(3, min_periods=1).mean().replace(0, np.nan)
        ) * (
            out["kfin_total_value"] / out["kfin_total_value"].rolling(5, min_periods=1).mean().replace(0, np.nan)
        )

    # DRAM 비중 변화 × 주가 수익률: DRAM 의존도 신호
    if "semi_dram_ratio" in out.columns and "return_1d" in out.columns:
        dram_ratio_delta = out["semi_dram_ratio"] - out["semi_dram_ratio"].shift(1)
        out["dram_dependency_signal"] = dram_ratio_delta.fillna(0) * np.sign(out["return_1d"].fillna(0))

    # 수출입 비율 × 최대행사가: 수출 강세 + 옵션 낙관 복합
    if "semi_trade_ratio" in out.columns and "kfin_max_strike" in out.columns:
        out["export_optimism_index"] = (
            out["semi_trade_ratio"] / out["semi_trade_ratio"].rolling(3, min_periods=1).mean().replace(0, np.nan)
        ) * (
            out["kfin_max_strike"] / out["kfin_max_strike"].rolling(5, min_periods=1).mean().replace(0, np.nan)
        )

    # === 시차(Lag) 기반 피처 ===
    if "kfin_atm_price" in out.columns:
        out["kfin_atm_lag1"] = out["kfin_atm_price"].shift(1)
        out["kfin_atm_delta_5d"] = out["kfin_atm_price"] - out["kfin_atm_price"].shift(5)

    if "semi_total_exp" in out.columns:
        out["semi_exp_ma3"] = out["semi_total_exp"].rolling(3, min_periods=1).mean()

    if "semi_dram_exp" in out.columns:
        out["semi_dram_ma3"] = out["semi_dram_exp"].rolling(3, min_periods=1).mean()

    if "kfin_active_ratio" in out.columns:
        out["kfin_active_ratio_ma5"] = out["kfin_active_ratio"].rolling(5, min_periods=1).mean()

    # === 변동성 피처 ===
    out["realized_vol_5d"] = out["return_1d"].rolling(5, min_periods=1).std()
    out["realized_vol_20d"] = out["return_1d"].rolling(20, min_periods=1).std()
    out["vol_ratio"] = out["realized_vol_5d"] / out["realized_vol_20d"].replace(0, np.nan)
    out["gap_from_ma20"] = (
        out["close"] - out["close"].rolling(20).mean()
    ) / out["close"].rolling(20).mean()

    # NaN 정리
    out = out.ffill().bfill()

    return out


for ticker in TICKERS:
    feature_marts[ticker] = create_derived_features(feature_marts[ticker])
    print(f"{TICKER_NAMES[ticker]} 파생 변수 포함: {feature_marts[ticker].shape[1]} 컬럼")
    print(f"  컬럼 목록: {list(feature_marts[ticker].columns)}")

삼성전자 파생 변수 포함: 47 컬럼
  컬럼 목록: ['close', 'return_1d', 'kfin_atm_price', 'kfin_active_count', 'kfin_mean_price', 'kfin_total_value', 'kfin_max_strike', 'kfin_active_ratio', 'semi_total_exp', 'semi_total_imp', 'semi_net_trade', 'semi_dram_exp', 'semi_dram_imp', 'semi_flash_exp', 'semi_flash_imp', 'semi_mcp_exp', 'semi_dram_ratio', 'semi_exp_mom', 'semi_dram_mom', 'semi_trade_ratio', 'usd_krw_rate', 'yfinance_nvda_close', 'yfinance_amd_close', 'yfinance_mu_close', 'yfinance_tsm_close', 'yfinance_asml_close', 'yfinance_sox_close', 'fred_dff', 'fred_dgs10', 'fred_dgs2', 'fred_t10y2y', 'fred_dfii10', 'fred_bamlh0a0hym2', 'finance_export_synergy', 'atm_dram_cross', 'trade_finance_compound', 'dram_dependency_signal', 'export_optimism_index', 'kfin_atm_lag1', 'kfin_atm_delta_5d', 'semi_exp_ma3', 'semi_dram_ma3', 'kfin_active_ratio_ma5', 'realized_vol_5d', 'realized_vol_20d', 'vol_ratio', 'gap_from_ma20']
SK하이닉스 파생 변수 포함: 47 컬럼
  컬럼 목록: ['close', 'return_1d', 'kfin_atm_price', 'kfin_active_count'

# 4. TimesFM 2.5 모델 로드

In [0]:
import importlib
import timesfm  # noqa: E402
importlib.reload(timesfm)

torch.set_float32_matmul_precision("high")

# HuggingFace 인증
_hf_token = vault.get_secret("hf-token") or os.environ.get("HF_TOKEN", "")
if _hf_token:
    os.environ["HF_TOKEN"] = _hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = _hf_token
    print(f"HF 토큰 설정 완료 (len={len(_hf_token)})")
else:
    print("[WARN] HF 토큰 없음")

model = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
model.compile(
    timesfm.ForecastConfig(
        max_context=256,         # 1년 ≈ 250 거래일
        max_horizon=128,         # 최대 예측 길이
        return_backcast=True,    # XReg 추론에 필수
    )
)
print("TimesFM 2.5 200M 모델 로드 완료")

HF 토큰 설정 완료 (len=37)


config.json:   0%|          | 0.00/475 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/925M [00:00<?, ?B/s]

TimesFM 2.5 200M 모델 로드 완료


# 5. Step 1 — Zero-shot Baseline 추론

> 반도체 주가 종가 시계열만 입력 → 순수 가격 패턴 기반 예측

In [0]:
inputs = []
for ticker in TICKERS:
    series = feature_marts[ticker]["close"].values.astype(np.float32)
    inputs.append(series)

_pb_raw, _qb_raw = model.forecast(
    horizon=HORIZON,
    inputs=inputs,
)

# return_backcast=True → backcast 포함된 전체 배열에서 forecast 부분만 추출
point_baseline = np.array(_pb_raw)[:, -HORIZON:]
quantile_baseline = np.array(_qb_raw)[:, -HORIZON:, :]

print(f"Baseline point: {point_baseline.shape} | quantile: {quantile_baseline.shape}")

for i, ticker in enumerate(TICKERS):
    last_price = inputs[i][-1]
    pred_final = point_baseline[i, -1]
    change_pct = (pred_final - last_price) / last_price * 100
    print(f"\n{TICKER_NAMES[ticker]}:")
    print(f"  현재가: {last_price:,.0f}")
    print(f"  {HORIZON}일 후 예측: {pred_final:,.0f} ({change_pct:+.2f}%)")
    print(f"  80% PI: [{quantile_baseline[i, -1, 1]:,.0f}, {quantile_baseline[i, -1, 9]:,.0f}]")

Baseline point: (2, 20) | quantile: (2, 20, 10)

삼성전자:
  현재가: 209,250
  20일 후 예측: 171,192 (-18.19%)
  80% PI: [135,939, 220,374]

SK하이닉스:
  현재가: 1,002,000
  20일 후 예측: 1,016,796 (+1.48%)
  80% PI: [720,648, 1,327,487]


# 6. Step 2 — XReg 공변량 추론

> 매크로/퀀트/감성 지표를 외부 회귀 변수(External Regressors)로 추가.
> 모델 파라미터를 재학습하지 않고, 추론 시점에만 공변량을 참고합니다.

In [0]:

def prepare_covariate(series_values, total_len):
    """공변량 배열을 context + horizon 길이에 맞추고, 부족분은 마지막 값으로 패딩합니다."""
    values = np.asarray(series_values, dtype=np.float32)
    # NaN 제거
    mask = np.isfinite(values)
    if not mask.all():
        valid = values[mask]
        if len(valid) == 0:
            return np.zeros(total_len, dtype=np.float32)
        values = np.interp(np.arange(len(values)), np.where(mask)[0], valid).astype(np.float32)
    if len(values) < total_len:
        pad = np.full(total_len - len(values), values[-1], dtype=np.float32)
        values = np.concatenate([values, pad])
    return values[:total_len]


# 공변량으로 사용할 컬럼 목록 (타겟 'close' 및 파생 수익률 제외)
EXCLUDE_COLS = {"close", "return_1d"}

# 각 종목별 공변량을 구성
dynamic_numerical = {}
for col in feature_marts[TICKERS[0]].columns:
    if col in EXCLUDE_COLS:
        continue
    arrays = []
    for ticker in TICKERS:
        mart = feature_marts[ticker]
        context_len = len(mart)
        total_len = context_len + HORIZON
        arrays.append(prepare_covariate(mart[col].values, total_len))
    dynamic_numerical[col] = arrays

# 요일 카테고리 (0=월 ~ 4=금)
dynamic_categorical = {}
day_of_week_arrays = []
for ticker in TICKERS:
    mart = feature_marts[ticker]
    dates = mart.index
    dow = [str(d.weekday()) for d in dates]
    # 미래 요일: 마지막 거래일부터 비즈니스 데이 기준 생성
    future_dates = pd.bdate_range(dates[-1] + pd.Timedelta(days=1), periods=HORIZON)
    dow_future = [str(d.weekday()) for d in future_dates]
    day_of_week_arrays.append(dow + dow_future)
dynamic_categorical["day_of_week"] = day_of_week_arrays

# 종목 구분 (Static)
static_categorical = {"ticker": list(TICKERS)}

print(f"Dynamic numerical covariates: {len(dynamic_numerical)} 개")
print(f"  컬럼: {list(dynamic_numerical.keys())}")

Dynamic numerical covariates: 45 개
  컬럼: ['kfin_atm_price', 'kfin_active_count', 'kfin_mean_price', 'kfin_total_value', 'kfin_max_strike', 'kfin_active_ratio', 'semi_total_exp', 'semi_total_imp', 'semi_net_trade', 'semi_dram_exp', 'semi_dram_imp', 'semi_flash_exp', 'semi_flash_imp', 'semi_mcp_exp', 'semi_dram_ratio', 'semi_exp_mom', 'semi_dram_mom', 'semi_trade_ratio', 'usd_krw_rate', 'yfinance_nvda_close', 'yfinance_amd_close', 'yfinance_mu_close', 'yfinance_tsm_close', 'yfinance_asml_close', 'yfinance_sox_close', 'fred_dff', 'fred_dgs10', 'fred_dgs2', 'fred_t10y2y', 'fred_dfii10', 'fred_bamlh0a0hym2', 'finance_export_synergy', 'atm_dram_cross', 'trade_finance_compound', 'dram_dependency_signal', 'export_optimism_index', 'kfin_atm_lag1', 'kfin_atm_delta_5d', 'semi_exp_ma3', 'semi_dram_ma3', 'kfin_active_ratio_ma5', 'realized_vol_5d', 'realized_vol_20d', 'vol_ratio', 'gap_from_ma20']


In [0]:
# XReg 추론 — xreg_mode="xreg + timesfm"
_xreg_result = model.forecast_with_covariates(
    inputs=inputs,
    dynamic_numerical_covariates=dynamic_numerical,
    dynamic_categorical_covariates=dynamic_categorical,
    static_categorical_covariates=static_categorical,
    xreg_mode="xreg + timesfm",
)

# TimesFM 2.5는 list 반환 → np.array 변환
point_xreg = np.array(_xreg_result[0])
quantile_xreg = np.array(_xreg_result[1])

print(f"XReg point shape: {point_xreg.shape} | quantile shape: {quantile_xreg.shape}")

for i, ticker in enumerate(TICKERS):
    last_price = inputs[i][-1]
    pred_base = point_baseline[i, -1]
    pred_xreg_val = point_xreg[i, -1]
    delta = pred_xreg_val - pred_base
    print(f"\n{TICKER_NAMES[ticker]}:")
    print(f"  Baseline {HORIZON}일 예측: {pred_base:,.0f}")
    print(f"  XReg     {HORIZON}일 예측: {pred_xreg_val:,.0f}")
    print(f"  매크로 충격 (Δ): {delta:+,.0f} ({delta / last_price * 100:+.2f}%)")
    if quantile_xreg.ndim == 3:
        print(f"  XReg 80% PI: [{quantile_xreg[i, -1, 1]:,.0f}, {quantile_xreg[i, -1, 9]:,.0f}]")

XReg point shape: (2, 20) | quantile shape: (2, 20, 10)

삼성전자:
  Baseline 20일 예측: 171,192
  XReg     20일 예측: 188,081
  매크로 충격 (Δ): +16,889 (+8.07%)
  XReg 80% PI: [177,697, 204,139]

SK하이닉스:
  Baseline 20일 예측: 1,016,796
  XReg     20일 예측: 963,410
  매크로 충격 (Δ): -53,386 (-5.33%)
  XReg 80% PI: [897,650, 1,028,982]


## 6-1. AI 해석 — 예측 결과 (Zero-shot vs XReg)

In [0]:
# GPT-4.1-mini로 Step 1 + Step 2 예측 결과를 투자 관점에서 해석
_forecast_summary_lines = [
    f"SENSE TimesFM 2.5 모델 — 향후 {HORIZON} 거래일(약 4주) 주가 예측 결과입니다.\n",
]
for i, ticker in enumerate(TICKERS):
    lp = inputs[i][-1]
    b_fin = point_baseline[i, -1]
    x_fin = point_xreg[i, -1]
    delta = x_fin - b_fin
    b_pct = (b_fin - lp) / lp * 100
    x_pct = (x_fin - lp) / lp * 100
    d_pct = delta / lp * 100
    _forecast_summary_lines.append(
        f"[{TICKER_NAMES[ticker]} ({ticker})]\n"
        f"  현재가: {lp:,.0f}원\n"
        f"  Zero-shot Baseline T+{HORIZON}: {b_fin:,.0f}원 ({b_pct:+.2f}%)\n"
        f"  XReg(매크로반영) T+{HORIZON}: {x_fin:,.0f}원 ({x_pct:+.2f}%)\n"
        f"  매크로 충격 Δ: {delta:+,.0f}원 ({d_pct:+.2f}%) "
        f"→ {'매크로가 주가를 끌어올리는 방향' if delta > 0 else '매크로가 주가를 억누르는 방향'}\n"
        f"  XReg 80% PI: [{quantile_xreg[i, -1, 1]:,.0f}, {quantile_xreg[i, -1, 9]:,.0f}]원\n"
    )

_forecast_prompt = (
    "\n".join(_forecast_summary_lines) + "\n위 예측 결과를 바탕으로:\n"
    "1) 두 종목의 단기 방향성과 매크로 환경의 영향을 종합 해석해 주세요.\n"
    "2) Zero-shot과 XReg 차이(매크로 충격)의 의미를 설명해 주세요.\n"
    "3) 투자자가 주목해야 할 핵심 포인트 2~3가지를 제시해 주세요.\n"
    "답변은 한국어로, 300자 이내로 핵심만 간결하게 작성해 주세요."
)

_forecast_interpretation = ask_gpt(_forecast_prompt, system_msg=_SYSTEM_MSG, max_tokens=600)
print("=" * 70)
print("🤖 AI 해석 — 예측 결과")
print("=" * 70)
print(_forecast_interpretation)

🤖 AI 해석 — 예측 결과
삼성전자는 매크로 반영 시 20거래일 후 -10.12% 하락 예상되나, 매크로 충격이 +8.07%로 주가 하락 폭을 완화합니다. SK하이닉스는 매크로 영향으로 -5.33% 하락 압박을 받아 XReg 기준 -3.85% 하락 전망입니다. Zero-shot은 순수 시계열 추세, XReg는 매크로 변수 반영해 현실적 변동성 반영 의미입니다. 투자자는 매크로 환경 변화, 삼성전자의 하락 완화 가능성, SK하이닉스의 매크로 리스크를 주목해야 합니다.


Trace(request_id=tr-d8b7f27b9fc6438a807af15e1f0b91bf)

# 7. 매크로 충격 분석 및 시각화

In [0]:
fig, axes = plt.subplots(len(TICKERS), 1, figsize=(14, 5 * len(TICKERS)), sharex=True)
if len(TICKERS) == 1:
    axes = [axes]

for i, ticker in enumerate(TICKERS):
    ax = axes[i]
    context_len = len(inputs[i])
    context_x = np.arange(context_len)
    forecast_x = np.arange(context_len, context_len + HORIZON)

    # 최근 60일 + 예측
    show_from = max(0, context_len - 60)

    ax.plot(
        context_x[show_from:],
        inputs[i][show_from:],
        color="black",
        linewidth=1.5,
        label="실제 종가",
    )

    # Baseline
    ax.plot(
        forecast_x,
        point_baseline[i],
        color="tab:blue",
        linewidth=1.5,
        linestyle="--",
        label="Baseline (Zero-shot)",
    )
    ax.fill_between(
        forecast_x,
        quantile_baseline[i, :, 1],
        quantile_baseline[i, :, 9],
        alpha=0.15,
        color="tab:blue",
        label="Baseline 80% PI",
    )

    # XReg
    ax.plot(
        forecast_x, point_xreg[i], color="tab:orange", linewidth=1.5, label="XReg (매크로 반영)"
    )
    ax.fill_between(
        forecast_x,
        quantile_xreg[i, :, 1],
        quantile_xreg[i, :, 9],
        alpha=0.15,
        color="tab:orange",
        label="XReg 80% PI",
    )

    ax.axvline(x=context_len - 0.5, color="gray", linestyle=":", alpha=0.5)
    ax.set_title(f"{TICKER_NAMES[ticker]} ({ticker}) — {HORIZON}일 예측", fontsize=13)
    ax.set_ylabel("종가")
    ax.legend(loc="upper left", fontsize=9)
    ax.grid(True, alpha=0.3)

plt.xlabel("거래일 인덱스")
plt.tight_layout()
plt.savefig("/tmp/timesfm_forecast_comparison.png", dpi=150)
plt.show()
print("시각화 저장: /tmp/timesfm_forecast_comparison.png")

/root/.ipykernel/1919/command-7235105398391575-3377408844:60: UserWarning: Glyph 51333 (\N{HANGUL SYLLABLE JONG}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/command-7235105398391575-3377408844:60: UserWarning: Glyph 44032 (\N{HANGUL SYLLABLE GA}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/command-7235105398391575-3377408844:60: UserWarning: Glyph 49340 (\N{HANGUL SYLLABLE SAM}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/command-7235105398391575-3377408844:60: UserWarning: Glyph 49457 (\N{HANGUL SYLLABLE SEONG}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/command-7235105398391575-3377408844:60: UserWarning: Glyph 51204 (\N{HANGUL SYLLABLE JEON}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/command-7235105398391575-3377408844:60: UserWarning: Glyph 51088 (\N{HANGUL SYLLABLE JA}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/command-72

시각화 저장: /tmp/timesfm_forecast_comparison.png


## 7-1. 매크로 충격 타임라인

In [0]:
macro_impact = point_xreg - point_baseline

fig, axes = plt.subplots(len(TICKERS), 1, figsize=(14, 4 * len(TICKERS)), sharex=True)
if len(TICKERS) == 1:
    axes = [axes]

for i, ticker in enumerate(TICKERS):
    ax = axes[i]
    days = np.arange(1, HORIZON + 1)
    impact = macro_impact[i]
    colors = ["tab:red" if v < 0 else "tab:green" for v in impact]
    ax.bar(days, impact, color=colors, alpha=0.7)
    ax.axhline(y=0, color="black", linewidth=0.5)
    ax.set_title(f"{TICKER_NAMES[ticker]} — 매크로 충격 (XReg − Baseline)", fontsize=12)
    ax.set_ylabel("충격 (원)")
    ax.grid(True, alpha=0.3, axis="y")

plt.xlabel("예측 일차 (T+n)")
plt.tight_layout()
plt.savefig("/tmp/timesfm_macro_impact.png", dpi=150)
plt.show()

/root/.ipykernel/1919/command-7235105398391577-1008512189:19: UserWarning: Glyph 52649 (\N{HANGUL SYLLABLE CUNG}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/command-7235105398391577-1008512189:19: UserWarning: Glyph 44201 (\N{HANGUL SYLLABLE GYEOG}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/command-7235105398391577-1008512189:19: UserWarning: Glyph 50896 (\N{HANGUL SYLLABLE WEON}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/command-7235105398391577-1008512189:19: UserWarning: Glyph 49340 (\N{HANGUL SYLLABLE SAM}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/command-7235105398391577-1008512189:19: UserWarning: Glyph 49457 (\N{HANGUL SYLLABLE SEONG}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/command-7235105398391577-1008512189:19: UserWarning: Glyph 51204 (\N{HANGUL SYLLABLE JEON}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/comma

# 8. 시나리오 분석 (What-If Simulation)

> 미래 공변량 값을 시나리오별로 교체하여 "만약 X가 발생하면?" 질문에 즉시 답변

In [0]:
# 현재 최신 값 가져오기
mart = feature_marts[TICKERS[0]]
current_vals = {col: mart[col].iloc[-1] for col in mart.columns}

SCENARIOS = {
    "금리 인하 (-50bp)": {
        "fred_dgs10": current_vals.get("fred_dgs10", 4.0) - 0.5,
        "fred_t10y2y": current_vals.get("fred_t10y2y", 0.0) + 0.3,
    },
    "현상 유지": {},
    "금리 인상 (+50bp)": {
        "fred_dgs10": current_vals.get("fred_dgs10", 4.0) + 0.5,
        "fred_t10y2y": current_vals.get("fred_t10y2y", 0.0) - 0.3,
    },
    "원화 약세 (+5%)": {
        "usd_krw_rate": current_vals.get("usd_krw_rate", 1400.0) * 1.05,
    },
    "반도체 수출 급증 (+20%)": {
        "semi_total_exp": current_vals.get("semi_total_exp", 300.0) * 1.20,
        "semi_dram_exp": current_vals.get("semi_dram_exp", 80.0) * 1.20,
    },
}

scenario_results = {}

for scenario_name, overrides in SCENARIOS.items():
    scenario_covariates = {}
    for key, arrays in dynamic_numerical.items():
        modified_arrays = []
        for arr in arrays:
            new_arr = arr.copy()
            if key in overrides:
                context_len = len(inputs[0])
                new_arr[context_len:] = overrides[key]
            modified_arrays.append(new_arr)
        scenario_covariates[key] = modified_arrays

    _p_sc, _q_sc = model.forecast_with_covariates(
        inputs=inputs,
        dynamic_numerical_covariates=scenario_covariates,
        dynamic_categorical_covariates=dynamic_categorical,
        static_categorical_covariates=static_categorical,
        xreg_mode="xreg + timesfm",
    )
    # list → np.array 변환
    scenario_results[scenario_name] = {
        "point": np.array(_p_sc),
        "quantiles": np.array(_q_sc),
    }
    print(f"  ✅ {scenario_name}")

print(f"\n총 {len(scenario_results)} 시나리오 추론 완료")

  ✅ 금리 인하 (-50bp)
  ✅ 현상 유지
  ✅ 금리 인상 (+50bp)
  ✅ 원화 약세 (+5%)
  ✅ 반도체 수출 급증 (+20%)

총 5 시나리오 추론 완료


In [0]:
# 시나리오별 최종 예측값 비교 테이블
rows = []
for scenario_name, result in scenario_results.items():
    for i, ticker in enumerate(TICKERS):
        last_price = inputs[i][-1]
        pred = result["point"][i, -1]
        change_pct = (pred - last_price) / last_price * 100
        rows.append(
            {
                "시나리오": scenario_name,
                "종목": TICKER_NAMES[ticker],
                "현재가": last_price,
                f"T+{HORIZON} 예측": pred,
                "변동률 (%)": round(change_pct, 2),
            }
        )

df_scenarios = pd.DataFrame(rows)
print(df_scenarios.to_string(index=False))

            시나리오     종목       현재가      T+20 예측  변동률 (%)
   금리 인하 (-50bp)   삼성전자  209250.0 1.739965e+05   -16.85
   금리 인하 (-50bp) SK하이닉스 1002000.0 8.832894e+05   -11.85
           현상 유지   삼성전자  209250.0 1.880808e+05   -10.12
           현상 유지 SK하이닉스 1002000.0 9.634104e+05    -3.85
   금리 인상 (+50bp)   삼성전자  209250.0 2.021652e+05    -3.39
   금리 인상 (+50bp) SK하이닉스 1002000.0 1.043531e+06     4.14
     원화 약세 (+5%)   삼성전자  209250.0 1.882971e+05   -10.01
     원화 약세 (+5%) SK하이닉스 1002000.0 9.646406e+05    -3.73
반도체 수출 급증 (+20%)   삼성전자  209250.0 1.894619e+05    -9.46
반도체 수출 급증 (+20%) SK하이닉스 1002000.0 9.712670e+05    -3.07


In [0]:
# 시나리오 Fan Chart
fig, axes = plt.subplots(1, len(TICKERS), figsize=(7 * len(TICKERS), 5))
if len(TICKERS) == 1:
    axes = [axes]

colors = ["tab:green", "tab:gray", "tab:red", "tab:purple", "tab:brown"]

for idx, ticker in enumerate(TICKERS):
    ax = axes[idx]
    forecast_x = np.arange(1, HORIZON + 1)

    for j, (name, result) in enumerate(scenario_results.items()):
        c = colors[j % len(colors)]
        ax.plot(forecast_x, result["point"][idx], label=name, color=c, linewidth=1.5)
        ax.fill_between(
            forecast_x,
            result["quantiles"][idx, :, 1],
            result["quantiles"][idx, :, 9],
            alpha=0.08,
            color=c,
        )

    ax.axhline(y=inputs[idx][-1], color="black", linestyle=":", alpha=0.5, label="현재가")
    ax.set_title(f"{TICKER_NAMES[ticker]} 시나리오 분석", fontsize=12)
    ax.set_xlabel(f"예측 일차 (T+1 ~ T+{HORIZON})")
    ax.set_ylabel("예측 종가")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/tmp/timesfm_scenario_analysis.png", dpi=150)
plt.show()

/root/.ipykernel/1919/command-7235105398391581-3496786114:30: UserWarning: Glyph 50696 (\N{HANGUL SYLLABLE YE}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/command-7235105398391581-3496786114:30: UserWarning: Glyph 52769 (\N{HANGUL SYLLABLE CEUG}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/command-7235105398391581-3496786114:30: UserWarning: Glyph 51068 (\N{HANGUL SYLLABLE IL}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/command-7235105398391581-3496786114:30: UserWarning: Glyph 52264 (\N{HANGUL SYLLABLE CA}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/command-7235105398391581-3496786114:30: UserWarning: Glyph 51333 (\N{HANGUL SYLLABLE JONG}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/command-7235105398391581-3496786114:30: UserWarning: Glyph 44032 (\N{HANGUL SYLLABLE GA}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/command-723510

## 8-1. AI 해석 — 시나리오 분석

In [0]:
# GPT-4.1-mini로 시나리오별 결과를 경제적 맥락에서 해석
_scenario_lines = [
    f"반도체 주가 TimesFM 시나리오 분석 결과 (향후 {HORIZON} 거래일):\n",
]
for name in SCENARIOS:
    row_parts = []
    for i, ticker in enumerate(TICKERS):
        pred = scenario_results[name]["point"][i, -1]
        chg = (pred - inputs[i][-1]) / inputs[i][-1] * 100
        row_parts.append(f"{TICKER_NAMES[ticker]} {chg:+.2f}%")
    _scenario_lines.append(f"  [{name}]: {' | '.join(row_parts)}")

_scenario_prompt = (
    "\n".join(_scenario_lines) + "\n\n위 5가지 시나리오 분석 결과를 바탕으로:\n"
    "1) 각 시나리오가 반도체 주가에 미치는 영향을 경제적 논리와 함께 설명해 주세요.\n"
    "2) 가장 위험한 시나리오와 가장 유리한 시나리오를 특정하고 그 이유를 서술해 주세요.\n"
    "3) 현재 투자자가 헤지해야 할 시나리오 1가지를 추천해 주세요.\n"
    "답변은 한국어로, 400자 이내로 작성해 주세요."
)

_scenario_interpretation = ask_gpt(_scenario_prompt, system_msg=_SYSTEM_MSG, max_tokens=700)
print("=" * 70)
print("🤖 AI 해석 — 시나리오 분석")
print("=" * 70)
print(_scenario_interpretation)

🤖 AI 해석 — 시나리오 분석
금리 인하 시 삼성전자(-16.85%)·SK하이닉스(-11.85%) 급락은 금리 인하가 경기 부양 기대에도 반도체 업황 둔화 우려를 자극하기 때문입니다. 금리 인상 시 SK하이닉스(+4.14%) 상승은 금융시장 안정과 달러 강세에 따른 수출 경쟁력 강화 영향입니다. 원화 약세와 반도체 수출 급증 시 주가 하락은 원자재 비용 상승과 공급망 불안 가능성 때문입니다. 가장 위험한 시나리오는 금리 인하로, 주가 낙폭이 최대이며 경기 둔화 우려가 반영됐습니다. 가장 유리한 시나리오는 금리 인상으로 SK하이닉스가 수혜를 입습니다. 투자자는 금리 인하 시나리오에 대비해 헤지 전략을 권합니다.


Trace(request_id=tr-223458eef3bc4c6ebe45515e17c95b3b)

# 9. XReg Attribution — 공변량 기여도 분석

> Leave-One-Out 방식: 공변량을 하나씩 제외하며 추론 → 예측 변화량 = 해당 변수의 기여도

In [0]:
attribution = {}
n_covariates = len(dynamic_numerical)

print(f"공변량 기여도 분석 시작 ({n_covariates}개 변수)...")
for cov_idx, cov_name in enumerate(dynamic_numerical.keys()):
    reduced = {k: v for k, v in dynamic_numerical.items() if k != cov_name}
    _p_reduced, _ = model.forecast_with_covariates(
        inputs=inputs,
        dynamic_numerical_covariates=reduced,
        dynamic_categorical_covariates=dynamic_categorical,
        static_categorical_covariates=static_categorical,
        xreg_mode="xreg + timesfm",
    )
    point_reduced = np.array(_p_reduced)
    attribution[cov_name] = float(np.mean(np.abs(point_xreg - point_reduced)))

    if (cov_idx + 1) % 5 == 0:
        print(f"  {cov_idx + 1}/{n_covariates} 완료...")

print("기여도 분석 완료")

공변량 기여도 분석 시작 (45개 변수)...
  5/45 완료...
  10/45 완료...
  15/45 완료...
  20/45 완료...
  25/45 완료...
  30/45 완료...
  35/45 완료...
  40/45 완료...
  45/45 완료...
기여도 분석 완료


In [0]:
# 기여도 상위 10개 시각화
sorted_attr = sorted(attribution.items(), key=lambda x: x[1], reverse=True)
top_n = 10
names = [a[0] for a in sorted_attr[:top_n]]
scores = [a[1] for a in sorted_attr[:top_n]]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(  # noqa: F841
    range(top_n), scores[::-1], color="steelblue", alpha=0.8
)
ax.set_yticks(range(top_n))
ax.set_yticklabels(names[::-1], fontsize=10)
ax.set_xlabel("기여도 (예측 변화량 평균)")
ax.set_title(f"XReg Attribution — 예측에 가장 큰 영향을 미친 변수 Top {top_n}", fontsize=12)
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig("/tmp/timesfm_xreg_attribution.png", dpi=150)
plt.show()

print("\n[XReg 기여도 Top 5]")
for name, score in sorted_attr[:5]:
    print(f"  {name}: {score:.4f}")


[XReg 기여도 Top 5]
  kfin_max_strike: 5961.8955
  yfinance_tsm_close: 5650.9575
  kfin_mean_price: 4267.2227
  export_optimism_index: 3800.0208
  usd_krw_rate: 3715.1301


/root/.ipykernel/1919/command-7235105398391586-4274572869:16: UserWarning: Glyph 44592 (\N{HANGUL SYLLABLE GI}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/command-7235105398391586-4274572869:16: UserWarning: Glyph 50668 (\N{HANGUL SYLLABLE YEO}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/command-7235105398391586-4274572869:16: UserWarning: Glyph 46020 (\N{HANGUL SYLLABLE DO}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/command-7235105398391586-4274572869:16: UserWarning: Glyph 50696 (\N{HANGUL SYLLABLE YE}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/command-7235105398391586-4274572869:16: UserWarning: Glyph 52769 (\N{HANGUL SYLLABLE CEUG}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/command-7235105398391586-4274572869:16: UserWarning: Glyph 48320 (\N{HANGUL SYLLABLE BYEON}) missing from current font.
  plt.tight_layout()
/root/.ipykernel/1919/command-7235

## 9-1. AI 해석 — 공변량 기여도 (XReg Attribution)

In [0]:
# GPT-4.1-mini로 상위 기여 변수의 의미와 투자 시사점을 해석
_attr_lines = [
    "TimesFM XReg Leave-One-Out Attribution 분석 결과 (예측에 영향을 가장 크게 미친 변수):\n",
]
for rank, (name, score) in enumerate(sorted_attr[:10], 1):
    _attr_lines.append(f"  {rank}위. {name}: 기여도 {score:.4f}")

_attr_prompt = (
    "\n".join(_attr_lines) + "\n\n위 변수 기여도 분석 결과를 바탕으로:\n"
    "1) 상위 3개 변수가 왜 반도체 주가 예측에 중요한지 경제적 논리를 설명해 주세요.\n"
    "2) 현재 시장 상황에서 이 변수들이 시사하는 리스크 또는 기회를 서술해 주세요.\n"
    "3) 향후 모니터링이 가장 중요한 변수 2가지와 그 이유를 제시해 주세요.\n"
    "답변은 한국어로, 350자 이내로 작성해 주세요."
)

_attr_interpretation = ask_gpt(_attr_prompt, system_msg=_SYSTEM_MSG, max_tokens=650)
print("=" * 70)
print("🤖 AI 해석 — XReg Attribution (핵심 영향 변수)")
print("=" * 70)
print(_attr_interpretation)

🤖 AI 해석 — XReg Attribution (핵심 영향 변수)
상위 3개 변수는 반도체 주가 예측에 핵심적입니다. kfin_max_strike는 옵션 최대 행사가로, 투자자 기대 변동성 및 시장 심리를 반영해 가격 변동성을 예측합니다. yfinance_tsm_close는 대표 반도체 기업 TSMC 주가로, 글로벌 반도체 수요와 공급 상황을 직접 반영합니다. kfin_mean_price는 평균 거래가격으로, 시장 내 실제 거래 강도와 투자자 신뢰도를 나타냅니다. 현재 글로벌 공급망 불안과 미·중 갈등 심화로 kfin_max_strike와 TSMC 주가 변동성이 커 리스크가 높으며, 수요 회복 시 기회도 존재합니다. 향후 kfin_max_strike와 yfinance_tsm_close를 집중 모니터링해야 하며, 이는 시장 심리 변화와 글로벌 수급 상황을 신속히 반영하기 때문입니다.


Trace(request_id=tr-9e2203ea4e96439abd59eef9be3a14f1)

# 10. Rolling Window Backtest

> 최근 3개월(60 거래일)을 5일 단위로 슬라이딩하며
> 20일 예측 → MAE, PI Coverage, 방향 정확도 평가

In [0]:
BACKTEST_HORIZON = 20
STEP = 5
TEST_WINDOW = 60  # 최근 60 거래일을 테스트 구간으로 사용

backtest_results = []

for ticker_idx, ticker in enumerate(TICKERS):
    series = inputs[ticker_idx]
    total_len = len(series)

    for start in range(total_len - TEST_WINDOW, total_len - BACKTEST_HORIZON, STEP):
        train = series[:start]
        actual = series[start : start + BACKTEST_HORIZON]

        if len(train) < 32 or len(actual) < BACKTEST_HORIZON:
            continue

        _p_bt, _q_bt = model.forecast(
            horizon=BACKTEST_HORIZON,
            inputs=[train],
        )
        # return_backcast=True → 마지막 HORIZON개만 forecast
        point_bt = np.array(_p_bt)[:, -BACKTEST_HORIZON:]
        quantile_bt = np.array(_q_bt)[:, -BACKTEST_HORIZON:, :]
        pred = point_bt[0, : len(actual)]

        mae = float(np.mean(np.abs(actual - pred)))
        rmse = float(np.sqrt(np.mean((actual - pred) ** 2)))
        mape = (
            float(np.mean(np.abs((actual - pred) / actual)) * 100)
            if np.all(actual != 0)
            else np.nan
        )

        # 80% PI Coverage
        in_band = (actual >= quantile_bt[0, : len(actual), 1]) & (
            actual <= quantile_bt[0, : len(actual), 9]
        )
        coverage_80 = float(np.mean(in_band))

        # 방향 정확도
        actual_dir = np.sign(np.diff(actual))
        pred_dir = np.sign(np.diff(pred))
        min_len = min(len(actual_dir), len(pred_dir))
        dir_accuracy = (
            float(np.mean(actual_dir[:min_len] == pred_dir[:min_len])) if min_len > 0 else np.nan
        )

        backtest_results.append(
            {
                "ticker": ticker,
                "name": TICKER_NAMES[ticker],
                "window_start": start,
                "mae": mae,
                "rmse": rmse,
                "mape": mape,
                "coverage_80": coverage_80,
                "directional_accuracy": dir_accuracy,
            }
        )

df_backtest = pd.DataFrame(backtest_results)
print(f"Backtest 완료: {len(df_backtest)} 윈도우")

Backtest 완료: 16 윈도우


In [0]:
# 종목별 Backtest 결과 요약
print("=" * 70)
print("Rolling Window Backtest 결과 요약")
print("=" * 70)

for ticker in TICKERS:
    sub = df_backtest[df_backtest["ticker"] == ticker]
    if sub.empty:
        continue
    print(f"\n{TICKER_NAMES[ticker]} ({ticker}):")
    print(f"  평균 MAE:           {sub['mae'].mean():,.2f}")
    print(f"  평균 RMSE:          {sub['rmse'].mean():,.2f}")
    print(f"  평균 MAPE:          {sub['mape'].mean():.2f}%")
    print(f"  80% PI Coverage:    {sub['coverage_80'].mean():.1%}")
    print(f"  방향 정확도:         {sub['directional_accuracy'].mean():.1%}")
    print(f"  (윈도우 수: {len(sub)})")

Rolling Window Backtest 결과 요약

삼성전자 (005930.KS):
  평균 MAE:           17,469.66
  평균 RMSE:          21,056.53
  평균 MAPE:          9.37%
  80% PI Coverage:    63.1%
  방향 정확도:         46.7%
  (윈도우 수: 8)

SK하이닉스 (000660.KS):
  평균 MAE:           77,722.58
  평균 RMSE:          93,804.22
  평균 MAPE:          8.41%
  80% PI Coverage:    83.1%
  방향 정확도:         51.3%
  (윈도우 수: 8)


# 11. Anomaly Detection — PI 밴드 이탈 체크

> 최근 실제 종가가 TimesFM 예측 밴드를 이탈했는지 확인 → 이상 이벤트 기록

In [0]:
anomaly_records = []

for i, ticker in enumerate(TICKERS):
    series = inputs[i]
    if len(series) <= HORIZON:
        continue

    context = series[:-HORIZON]
    recent_actual = series[-HORIZON:]

    _p_ad, _q_ad = model.forecast(horizon=HORIZON, inputs=[context])
    # return_backcast=True → 마지막 HORIZON개만 forecast
    point_ad = np.array(_p_ad)[:, -HORIZON:]
    quantile_ad = np.array(_q_ad)[:, -HORIZON:, :]

    lower_80 = quantile_ad[0, :, 1]  # 10th percentile
    upper_80 = quantile_ad[0, :, 9]  # 90th percentile
    dates = feature_marts[ticker].index[-HORIZON:]

    for j in range(HORIZON):
        actual_val = recent_actual[j]
        if actual_val < lower_80[j]:
            anomaly_records.append(
                {
                    "ticker": ticker,
                    "name": TICKER_NAMES[ticker],
                    "date": dates[j],
                    "actual": actual_val,
                    "lower_80": lower_80[j],
                    "upper_80": upper_80[j],
                    "type": "하방 이탈",
                    "severity": "CRITICAL",
                    "deviation_pct": (actual_val - lower_80[j]) / lower_80[j] * 100,
                }
            )
        elif actual_val > upper_80[j]:
            anomaly_records.append(
                {
                    "ticker": ticker,
                    "name": TICKER_NAMES[ticker],
                    "date": dates[j],
                    "actual": actual_val,
                    "lower_80": lower_80[j],
                    "upper_80": upper_80[j],
                    "type": "상방 이탈",
                    "severity": "WARNING",
                    "deviation_pct": (actual_val - upper_80[j]) / upper_80[j] * 100,
                }
            )

if anomaly_records:
    df_anomalies = pd.DataFrame(anomaly_records)
    print(f"이상 이벤트 {len(df_anomalies)}건 감지:")
    print(df_anomalies.to_string(index=False))
else:
    df_anomalies = pd.DataFrame()
    print("최근 구간에서 이상 이벤트 없음 (80% PI 내)")

최근 구간에서 이상 이벤트 없음 (80% PI 내)


# 12. 결과 PostgreSQL 적재

In [0]:
# 예측 결과를 DataFrame으로 구조화
forecast_rows = []
for i, ticker in enumerate(TICKERS):
    last_date = feature_marts[ticker].index[-1]
    future_dates = pd.bdate_range(last_date + pd.Timedelta(days=1), periods=HORIZON)

    for j in range(HORIZON):
        forecast_rows.append(
            {
                "ticker": ticker,
                "forecast_date": future_dates[j],
                "base_date": last_date,
                "horizon_day": j + 1,
                "baseline_point": float(point_baseline[i, j]),
                "baseline_q10": float(quantile_baseline[i, j, 1]),
                "baseline_q90": float(quantile_baseline[i, j, 9]),
                "xreg_point": float(point_xreg[i, j]),
                "xreg_q10": float(quantile_xreg[i, j, 1]),
                "xreg_q90": float(quantile_xreg[i, j, 9]),
                "macro_impact": float(point_xreg[i, j] - point_baseline[i, j]),
            }
        )

df_forecast = pd.DataFrame(forecast_rows)
print(f"예측 결과: {len(df_forecast)} rows")
print(df_forecast.head(10).to_string(index=False))

예측 결과: 40 rows
   ticker forecast_date  base_date  horizon_day  baseline_point  baseline_q10  baseline_q90    xreg_point      xreg_q10      xreg_q90  macro_impact
005930.KS    2026-04-09 2026-04-08            1   203543.781250 187950.859375 219109.593750 203178.500000 191757.656250 214640.593750   -365.281250
005930.KS    2026-04-10 2026-04-08            2   201110.312500 180313.500000 223660.593750 199240.625000 186816.359375 212187.375000  -1869.687500
005930.KS    2026-04-13 2026-04-08            3   198041.984375 174029.937500 225437.531250 196649.718750 183773.984375 210740.687500  -1392.265625
005930.KS    2026-04-14 2026-04-08            4   194214.109375 167360.406250 224586.437500 195545.968750 182978.250000 210366.515625   1331.859375
005930.KS    2026-04-15 2026-04-08            5   190411.000000 161547.093750 225370.531250 194555.968750 181748.312500 210167.828125   4144.968750
005930.KS    2026-04-16 2026-04-08            6   188167.593750 157726.937500 223535.968750 19347

In [0]:
# PostgreSQL 적재
# 운영 환경에서는 아래 주석을 해제하고 실행

# conn = vault.get_pg_connection()
# df_forecast.to_sql(
#     "fact_timesfm_forecast",
#     conn,
#     if_exists="append",
#     index=False,
#     method="multi",
# )
# print(f"PostgreSQL 적재 완료: {len(df_forecast)} rows → fact_timesfm_forecast")

# XReg Attribution 결과도 적재
# df_attr = pd.DataFrame([
#     {"covariate": k, "attribution_score": v, "base_date": str(last_date)}
#     for k, v in sorted_attr
# ])
# df_attr.to_sql("fact_timesfm_attribution", conn, if_exists="append", index=False)
# print(f"Attribution 적재 완료: {len(df_attr)} rows")

# 13. 실행 결과 요약

> 이 노트북의 출력물을 ADF 배치 파이프라인에
> 포함하여 매일 자동 실행할 수 있습니다.

In [0]:
print("=" * 70)
print("SENSE × TimesFM 2.5 — 추론 결과 요약")
print("=" * 70)

for i, ticker in enumerate(TICKERS):
    last_price = inputs[i][-1]
    base_final = point_baseline[i, -1]
    xreg_final = point_xreg[i, -1]
    impact = xreg_final - base_final

    print(f"\n{'─' * 50}")
    print(f"📊 {TICKER_NAMES[ticker]} ({ticker})")
    print(f"  현재가:              {last_price:>12,.0f}")
    base_pct = (base_final - last_price) / last_price * 100
    print(f"  Baseline T+{HORIZON}:     {base_final:>12,.0f}  ({base_pct:+.2f}%)")
    xreg_pct = (xreg_final - last_price) / last_price * 100
    print(f"  XReg     T+{HORIZON}:     {xreg_final:>12,.0f}  ({xreg_pct:+.2f}%)")
    print(f"  매크로 충격 (Δ):     {impact:>+12,.0f}  ({impact / last_price * 100:+.2f}%)")

    # Backtest
    bt = df_backtest[df_backtest["ticker"] == ticker]
    if not bt.empty:
        print(f"  Backtest MAE:        {bt['mae'].mean():>12,.2f}")
        print(f"  Backtest Coverage:   {bt['coverage_80'].mean():>11.1%}")
        print(f"  방향 정확도:         {bt['directional_accuracy'].mean():>11.1%}")

    # Attribution Top 3
    print(f"  Top 3 영향 변수:     {', '.join([a[0] for a in sorted_attr[:3]])}")

# 시나리오 요약
print(f"\n{'─' * 50}")
print("📋 시나리오 분석 결과:")
for name in SCENARIOS:
    for i, ticker in enumerate(TICKERS):
        pred = scenario_results[name]["point"][i, -1]
        change = (pred - inputs[i][-1]) / inputs[i][-1] * 100
        print(f"  [{name}] {TICKER_NAMES[ticker]}: {change:+.2f}%")

# Anomaly
if anomaly_records:
    print(f"\n⚠️ 이상 이벤트: {len(anomaly_records)}건 감지됨")
else:
    print("\n✅ 이상 이벤트: 없음")

print(f"\n{'=' * 70}")
print("전략 문서: ref/TimesFM.md")
print("다음 단계: XGBoost/LightGBM 분류 결과와 교차 합의(Consensus) 판정")

SENSE × TimesFM 2.5 — 추론 결과 요약

──────────────────────────────────────────────────
📊 삼성전자 (005930.KS)
  현재가:                   209,250
  Baseline T+20:          171,192  (-18.19%)
  XReg     T+20:          188,081  (-10.12%)
  매크로 충격 (Δ):          +16,889  (+8.07%)
  Backtest MAE:           17,469.66
  Backtest Coverage:         63.1%
  방향 정확도:               46.7%
  Top 3 영향 변수:     kfin_max_strike, yfinance_tsm_close, kfin_mean_price

──────────────────────────────────────────────────
📊 SK하이닉스 (000660.KS)
  현재가:                 1,002,000
  Baseline T+20:        1,016,796  (+1.48%)
  XReg     T+20:          963,410  (-3.85%)
  매크로 충격 (Δ):          -53,386  (-5.33%)
  Backtest MAE:           77,722.58
  Backtest Coverage:         83.1%
  방향 정확도:               51.3%
  Top 3 영향 변수:     kfin_max_strike, yfinance_tsm_close, kfin_mean_price

──────────────────────────────────────────────────
📋 시나리오 분석 결과:
  [금리 인하 (-50bp)] 삼성전자: -16.85%
  [금리 인하 (-50bp)] SK하이닉스: -11.85%
  [현상 유지] 삼성전자: -10.1

# 14. AI 종합 투자 의견

In [0]:
# GPT-4.1-mini가 모든 분석 결과를 종합하여 최종 투자 의견을 생성
_final_lines = [
    "SENSE TimesFM 전체 분석 종합 요약:\n",
    f"분석 대상: {', '.join(f'{TICKER_NAMES[t]}({t})' for t in TICKERS)}",
    f"예측 기간: 향후 {HORIZON} 거래일\n",
]

# 예측 방향성
for i, ticker in enumerate(TICKERS):
    lp = inputs[i][-1]
    x_fin = point_xreg[i, -1]
    x_pct = (x_fin - lp) / lp * 100
    delta_pct = (point_xreg[i, -1] - point_baseline[i, -1]) / lp * 100
    _final_lines.append(
        f"[{TICKER_NAMES[ticker]}] XReg 예측: {x_pct:+.2f}% | 매크로 충격: {delta_pct:+.2f}%"
    )

# Attribution Top 3
_final_lines.append(f"\n핵심 영향 변수 Top 3: {', '.join([a[0] for a in sorted_attr[:3]])}")

# 시나리오 요약 (최선/최악)
_sc_changes = {}
for name in SCENARIOS:
    avg_chg = np.mean(
        [
            (scenario_results[name]["point"][i, -1] - inputs[i][-1]) / inputs[i][-1] * 100
            for i in range(len(TICKERS))
        ]
    )
    _sc_changes[name] = avg_chg
_best_sc = max(_sc_changes, key=_sc_changes.get)
_worst_sc = min(_sc_changes, key=_sc_changes.get)
_final_lines.append(f"최선 시나리오: [{_best_sc}] 평균 {_sc_changes[_best_sc]:+.2f}%")
_final_lines.append(f"최악 시나리오: [{_worst_sc}] 평균 {_sc_changes[_worst_sc]:+.2f}%")

# Backtest 지표
if not df_backtest.empty:
    for ticker in TICKERS:
        sub = df_backtest[df_backtest["ticker"] == ticker]
        if not sub.empty:
            _final_lines.append(
                f"[{TICKER_NAMES[ticker]}] Backtest — "
                f"MAE: {sub['mae'].mean():,.0f} | "
                f"방향 정확도: {sub['directional_accuracy'].mean():.1%} | "
                f"80% PI Coverage: {sub['coverage_80'].mean():.1%}"
            )

# 이상 감지
_final_lines.append(
    f"\n이상 이벤트: {'없음' if not anomaly_records else f'{len(anomaly_records)}건 감지'}"
)

_final_prompt = (
    "\n".join(_final_lines)
    + "\n\n위 SENSE 전체 분석을 종합하여 다음 형식으로 최종 투자 의견을 작성해 주세요:\n\n"
    "★ 종합 시장 진단 (2문장)\n"
    "★ 삼성전자 투자 의견 및 근거 (2문장)\n"
    "★ SK하이닉스 투자 의견 및 근거 (2문장)\n"
    "★ 핵심 리스크 요인 (2가지)\n"
    "★ 핵심 기회 요인 (2가지)\n"
    "★ 단기 모니터링 포인트 (2가지)\n\n"
    "답변은 한국어로, 실제 애널리스트 보고서 수준으로 작성해 주세요. "
    "단, 투자 손실에 대한 책임 면책 문구를 마지막에 한 줄 추가해 주세요."
)

_final_interpretation = ask_gpt(_final_prompt, system_msg=_SYSTEM_MSG, max_tokens=1500)
print("=" * 70)
print("🤖 AI 종합 투자 의견 (SENSE × TimesFM × GPT-4.1-mini)")
print("=" * 70)
print(_final_interpretation)
print(f"\n{'=' * 70}")

🤖 AI 종합 투자 의견 (SENSE × TimesFM × GPT-4.1-mini)
★ 종합 시장 진단  
향후 20거래일 반도체 업종은 매크로 변수에 따른 변동성이 확대될 전망입니다. 특히 금리 정책 변화에 민감하게 반응하며, 전반적으로 불확실성이 상존하는 국면입니다.  

★ 삼성전자 투자 의견 및 근거  
삼성전자는 XReg 모델 기준 약 -10.12% 하락 압력이 예상되나, 매크로 충격(+8.07%)이 이를 일부 상쇄할 가능성이 있습니다. 다만, 방향 정확도가 46.7%로 낮아 단기 변동성에 유의하며 보수적 접근이 필요합니다.  

★ SK하이닉스 투자 의견 및 근거  
SK하이닉스는 XReg 예측 -3.85%와 매크로 충격 -5.33%가 모두 부정적 신호를 보내고 있어 단기 조정 가능성이 큽니다. Backtest 결과 방향 정확도 51.3%와 80% PI 커버리지 83.1%로 예측 신뢰도는 삼성전자 대비 다소 높으나, 리스크 관리가 중요합니다.  

★ 핵심 리스크 요인  
1) 금리 인하 시나리오(-50bp)에서 평균 -14.35% 급락 가능성으로 인한 투자 심리 위축  
2) 글로벌 반도체 수요 둔화 및 지정학적 불확실성 확대에 따른 실적 변동성 증가  

★ 핵심 기회 요인  
1) 금리 인상(+50bp) 시 평균 +0.38% 상승 효과를 통한 반도체 업종 내 상대적 강세 가능성  
2) 주요 기술 지표(kfin_max_strike, yfinance_tsm_close, kfin_mean_price)의 긍정적 변화 시 단기 반등 모멘텀 확보  

★ 단기 모니터링 포인트  
1) 한국 및 미국 금리 정책 발표 및 시장 반응 추이  
2) 글로벌 반도체 수요 지표 및 주요 고객사(IT 기업) 실적 발표 상황  

*본 보고서는 참고용이며, 투자 판단에 따른 손실에 대해 당사는 책임을 지지 않습니다.*



Trace(request_id=tr-c7b1fc505f8f4d28a79facc672b3c408)